# Lezione 9 · Pratica: reti generative avversarie (GAN)

In questa pratica completi autonomamente il codice di una rete generativa avversaria (GAN, dall'inglese *Generative Adversarial Network*), l'architettura vista nella Lezione 9 del corso "Advanced Deep Learning".

## Cosa farai in questa pratica

- Costruisci le due reti che formano una GAN: il generatore e il discriminatore, con strati lineari in PyTorch
- Completi il ciclo di addestramento: l'aggiornamento del discriminatore e l'aggiornamento del generatore con la perdita non saturante
- Osservi in un grafico se il generatore impara a coprire tutta la distribuzione dei dati reali o se cade in un collasso di modalità (*mode collapse*)

## Prerequisito

Prima di iniziare, assicurati di aver seguito la Lezione 9 in diretta (o di aver rivisto la registrazione) e di aver letto il capitolo del manuale sulle GAN. Questa pratica presuppone che tu conosca già l'obiettivo minimax, il ruolo del generatore e del discriminatore, e perché si usa la perdita non saturante.

Ogni esercizio ha lo scheletro del codice già scritto. Completa soltanto le parti contrassegnate con `# TODO`. Sotto ogni esercizio trovi una soluzione collassata: prova a risolverlo da solo prima di aprirla.

## Esercizio 1 · Il generatore e il discriminatore

### Il caso

Lavori a un progetto con un dataset molto piccolo: hai solo una manciata di misurazioni di due sensori che, rappresentate su un piano, formano due gruppi ben distinti (un mix di due gaussiane). Vuoi addestrare una GAN su questi dati per generare esempi sintetici aggiuntivi e aumentare così il set di addestramento prima di usarlo in un altro modello.

### Cosa manca

La cella seguente contiene già:

- La funzione `generar_datos_reales`, che simula il tuo piccolo dataset: due gaussiane centrate in `(-2.5, 0)` e `(2.5, 0)`
- Il metodo `forward` di entrambe le classi

Ti manca da completare la definizione degli strati di `Generador` e `Discriminador` dentro `nn.Sequential`.

### Suggerimenti

- Il generatore riceve un vettore di rumore di `latent_dim` dimensioni e deve restituire un punto in due dimensioni: l'ultimo strato lineare deve terminare con 2 uscite.
- Il discriminatore riceve un punto in due dimensioni e restituisce un unico logit (non applicare la sigmoide: la funzione di perdita che userai nell'Esercizio 2 la include già).
- Usa `nn.LeakyReLU(0.2)` come attivazione tra gli strati lineari, come nell'esempio della Lezione 9.
- Due strati nascosti di dimensione `hidden_dim` bastano per questo problema così semplice.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Datos reales: mezcla de dos gaussianas separadas en el plano (tu dataset pequeño)
def generar_datos_reales(n_muestras, centros=((-2.5, 0.0), (2.5, 0.0)), std=0.4):
    centros = np.array(centros)
    indices = np.random.randint(0, len(centros), size=n_muestras)
    puntos = centros[indices] + std * np.random.randn(n_muestras, 2)
    return torch.tensor(puntos, dtype=torch.float32)

LATENT_DIM = 2

In [ ]:
class Generador(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.red = nn.Sequential(
            # TODO: define las capas del generador.
            # Estructura esperada: Linear(latent_dim -> hidden_dim), LeakyReLU(0.2),
            # Linear(hidden_dim -> hidden_dim), LeakyReLU(0.2), Linear(hidden_dim -> 2)
            ...
        )

    def forward(self, z):
        return self.red(z)


class Discriminador(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.red = nn.Sequential(
            # TODO: define las capas del discriminador.
            # Estructura esperada: Linear(2 -> hidden_dim), LeakyReLU(0.2),
            # Linear(hidden_dim -> hidden_dim), LeakyReLU(0.2), Linear(hidden_dim -> 1)
            # La última capa no lleva activación: devuelve un logit
            ...
        )

    def forward(self, x):
        return self.red(x)


generador = Generador().to(device)
discriminador = Discriminador().to(device)

print(generador)
print(discriminador)

<details>
<summary>💡 Vedi soluzione</summary>

```python
class Generador(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z):
        return self.red(z)


class Discriminador(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),  # logit, senza sigmoide
        )

    def forward(self, x):
        return self.red(x)


generador = Generador().to(device)
discriminador = Discriminador().to(device)

print(generador)
print(discriminador)
```
</details>

## Esercizio 2 · Il ciclo di addestramento

### Il caso

Hai già definito le due reti. Ora ti manca la parte che le fa competere: il ciclo di addestramento che alterna l'aggiornamento del discriminatore e quello del generatore, come hai visto nella Lezione 9.

### Cosa manca

La cella seguente contiene già:

- La funzione di perdita `criterion` (`BCEWithLogitsLoss`, che combina sigmoide ed entropia incrociata binaria)
- Gli ottimizzatori `opt_g` e `opt_d`
- Il campionamento dei dati reali, del rumore e dei dati falsi a ogni passo
- Le etichette `etiquetas_reales` e `etiquetas_falsas`
- Il resto del ciclo: `zero_grad`, `backward`, `step` e la stampa dei progressi

Ti mancano da completare due blocchi:

1. **Aggiornamento del discriminatore**: calcola `loss_d_real` (la perdita del discriminatore sui dati reali, confrontati con `etiquetas_reales`), `loss_d_fake` (la perdita sui dati falsi, confrontati con `etiquetas_falsas`) e sommale in `loss_d`
2. **Aggiornamento del generatore**: calcola `loss_g` con la perdita non saturante, cioè confrontando l'uscita del discriminatore su `datos_falsos` con `etiquetas_reales` (il generatore vuole ingannare il discriminatore, non farsi riconoscere)

### Suggerimenti

- `criterion` riceve prima le previsioni del discriminatore e poi le etichette: `criterion(discriminador(...), etiquetas)`
- Nel passo del discriminatore, i dati falsi vengono generati con `.detach()`, già scritto per te: così il generatore non accumula gradiente in quell'aggiornamento
- Nel passo del generatore, campiona di nuovo il rumore e genera di nuovo i dati falsi senza `.detach()`, perché stavolta il gradiente deve arrivare fino al generatore
- Ricorda il perché della perdita non saturante: minimizzare `log(1 - D(G(z)))` dà gradienti quasi piatti all'inizio, quindi in pratica si massimizza `log(D(G(z)))`, che in codice equivale a confrontare con `etiquetas_reales`

In [ ]:
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(generador.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminador.parameters(), lr=2e-4, betas=(0.5, 0.999))

BATCH_SIZE = 256
N_STEPS = 3000

for paso in range(1, N_STEPS + 1):
    # ---------- 1) Actualiza el discriminador ----------
    datos_reales = generar_datos_reales(BATCH_SIZE).to(device)
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    datos_falsos = generador(z).detach()  # detach: el generador no se actualiza en este paso

    etiquetas_reales = torch.ones(BATCH_SIZE, 1, device=device)
    etiquetas_falsas = torch.zeros(BATCH_SIZE, 1, device=device)

    # TODO: calcula loss_d_real (pérdida sobre datos_reales frente a etiquetas_reales),
    # loss_d_fake (pérdida sobre datos_falsos frente a etiquetas_falsas) y súmalas en loss_d
    loss_d_real = ...
    loss_d_fake = ...
    loss_d = ...

    opt_d.zero_grad()
    loss_d.backward()
    opt_d.step()

    # ---------- 2) Actualiza el generador (pérdida no saturante) ----------
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    datos_falsos = generador(z)

    # TODO: calcula loss_g comparando discriminador(datos_falsos) con etiquetas_reales
    loss_g = ...

    opt_g.zero_grad()
    loss_g.backward()
    opt_g.step()

    if paso % 500 == 0:
        print(f"paso {paso:5d} | pérdida D: {loss_d.item():.3f} | pérdida G: {loss_g.item():.3f}")

In [ ]:
# Visualización: datos reales frente a muestras generadas, con la GAN ya entrenada
with torch.no_grad():
    muestra_real = generar_datos_reales(1000).numpy()
    z_vis = torch.randn(1000, LATENT_DIM, device=device)
    muestra_generada = generador(z_vis).cpu().numpy()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(muestra_real[:, 0], muestra_real[:, 1], s=10, alpha=0.5, label="Datos reales")
ax.scatter(muestra_generada[:, 0], muestra_generada[:, 1], s=10, alpha=0.5, label="Muestras generadas")
ax.set_title("GAN entrenada: datos reales frente a datos generados")
ax.legend()
ax.set_aspect("equal")
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(generador.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminador.parameters(), lr=2e-4, betas=(0.5, 0.999))

BATCH_SIZE = 256
N_STEPS = 3000

for paso in range(1, N_STEPS + 1):
    # ---------- 1) Aggiorna il discriminatore ----------
    datos_reales = generar_datos_reales(BATCH_SIZE).to(device)
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    datos_falsos = generador(z).detach()  # detach: il generatore non si aggiorna in questo passo

    etiquetas_reales = torch.ones(BATCH_SIZE, 1, device=device)
    etiquetas_falsas = torch.zeros(BATCH_SIZE, 1, device=device)

    # Perdita del discriminatore: classificare bene i reali e classificare bene i falsi
    loss_d_real = criterion(discriminador(datos_reales), etiquetas_reales)
    loss_d_fake = criterion(discriminador(datos_falsos), etiquetas_falsas)
    loss_d = loss_d_real + loss_d_fake

    opt_d.zero_grad()
    loss_d.backward()
    opt_d.step()

    # ---------- 2) Aggiorna il generatore (perdita non saturante) ----------
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    datos_falsos = generador(z)

    # Il generatore "vuole" che il discriminatore classifichi i suoi campioni come reali
    loss_g = criterion(discriminador(datos_falsos), etiquetas_reales)

    opt_g.zero_grad()
    loss_g.backward()
    opt_g.step()

    if paso % 500 == 0:
        print(f"passo {paso:5d} | perdita D: {loss_d.item():.3f} | perdita G: {loss_g.item():.3f}")
```
</details>

## Per concludere

Hai completato una GAN intera: hai definito il generatore e il discriminatore, e programmato il gioco minimax che li mette l'uno contro l'altro passo dopo passo. Guarda il grafico finale: se il generatore copre solo una delle due gaussiane, hai appena assistito in prima persona a un collasso di modalità, lo stesso problema visto nella Lezione 9.

Prima di chiudere il notebook, torna a `N_STEPS` e provalo con un valore molto più basso, per esempio 300. Confronta il risultato con la tua esecuzione precedente e verifica quanto addestramento serve a questa GAN così piccola per coprire le due modalità della distribuzione.

**Condividi il tuo grafico finale nel canale della lezione e racconta se il tuo generatore ha subito un collasso di modalità.**